In [1]:
import os
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

c:\Users\hp\LLM_ENGINEERING\agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)
open_route_key = os.getenv("OPENROUTER_API_KEY")

if open_route_key:
    print(f"OpenAI API Key exists and begins {open_route_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"


OpenAI API Key exists and begins sk-or-v1


In [5]:
openrouter_url = "https://openrouter.ai/api/v1"
client = OpenAI(api_key=open_route_key,base_url=openrouter_url)

In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [6]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_message}]+history+[{"role":"user","content":message}]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=1500
    )
    
    return response.choices[0].message.content

In [7]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [12]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_flight_ticket_price(destination_city):
    print(f"Tool Call for city ${destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")    
    return f"The price of a ticket to {destination_city} is {price}"


In [13]:
get_flight_ticket_price("London")

Tool Call for city $London


'The price of a ticket to London is $799'

In [17]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_flight_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [18]:
tools =[{"type":"function","function":price_function}]

In [19]:
tools

[{'type': 'function',
  'function': {'name': 'get_flight_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]